In [ ]:
''' creating from adjacent_schannel_study_siglepair
'''

In [ ]:
print("Running ")

In [ ]:
%load_ext autoreload
%autoreload 2

import importlib
import waffles.np04_analysis.lightyield_vs_energy.scripts.utils as utils_module
from waffles.np04_analysis.lightyield_vs_energy.scripts.utils import *
import ast
import copy


In [ ]:
energy_dict =  {1: 0, 2: 118, 3: 119, 5: 175, 7: 253} # Crossing point between langauss and gaussian distribution - to use for beam event selection trigger 

energies = [1,2,3,5,7] #[1,2,3,5,7]

apa_studied = ['1'] #['1', '2']

input_folder = f"/afs/cern.ch/work/a/anbalbon/private/waffles/src/waffles/np04_analysis/lightyield_vs_energy/output"
output_folder= f"{input_folder}/adjacent_chanels_study"


In [ ]:
adjacent_channels_apa1_map , adjacent_channels_apa2_map, _ = adjacent_channel_info()

adiacent_channels_map = {}
if '1' in apa_studied:
    adiacent_channels_map['1'] = adjacent_channels_apa1_map
if '2' in apa_studied:
    adiacent_channels_map['2'] = adjacent_channels_apa2_map

# adiacent_channels_map = {'1': [[{'apa' : 1, 'end': 104, 'ch': 15},{'apa' : 1, 'end': 104, 'ch': 12}]]}
# adiacent_channels_map = {'1': [[{'apa' : 1, 'end': 104, 'ch': 15},{'apa' : 1, 'end': 104, 'ch': 12}], [{'apa' : 1, 'end': 104, 'ch': 2},{'apa' : 1, 'end': 104, 'ch': 0}],[{'apa' : 1, 'end': 104, 'ch': 3},{'apa' : 1, 'end': 104, 'ch': 4}]], '2' : [[{'apa' : 2, 'end': 109, 'ch': 37},{'apa' : 2, 'end': 109, 'ch': 35}], [{'apa' : 2, 'end': 109, 'ch': 31},{'apa' : 2, 'end': 109, 'ch': 33}], [{'apa' : 2, 'end': 109, 'ch': 34},{'apa' : 2, 'end': 109, 'ch': 36}], [{'apa' : 2, 'end': 109, 'ch': 2},{'apa' : 2, 'end': 109, 'ch': 0}] ]}    


In [ ]:
all_data = {}

for energy in energies: 
    apa1_mean_pe_threshold = energy_dict[energy]

    apa12_energy_folder = f"{input_folder}/apa1_vs_apa2/{energy}GeV"

    merged_dict = load_energy_json_dict(energy, apa12_energy_folder, output_folder)


    # SELECTING TRIGGERS INDEXES WHERE APA1 MEAN PE > THRESHOLD
    dic_trigger_index = {}
    for _to_, to_dict in merged_dict.items(): # looking at each "0_to_10", "10_to_20", ...
        index_list = []
        for i, trigger_mean_pe in enumerate(to_dict['1']['mean']): # looking at each trigger
            try:
                if trigger_mean_pe > apa1_mean_pe_threshold: # checking if the mean photoelectrons in APA1 is above threshold
                    index_list.append(i)
            except:
                continue
        dic_trigger_index[_to_] = index_list # saving the list of trigger indexes that passed the threshold for this "_to_"


   

    adiacent_channels_dict = {}

    for apa, adiacent_channels_apa in adiacent_channels_map.items():
        adiacent_channels_apa_list = []

        for ch_pairs in adiacent_channels_apa:

            ch_pair_info = copy.deepcopy(ch_pairs)

            for i in range(2):
                ch_pair_info[i]['NPE'] = []
                ch_pair_info[i]['NPE_err'] = []

            apa1 = str(ch_pairs[0]['apa'])
            end1 = str(ch_pairs[0]['end'])
            ch1  = str(ch_pairs[0]['ch'])

            apa2 = str(ch_pairs[1]['apa'])
            end2 = str(ch_pairs[1]['end'])
            ch2  = str(ch_pairs[1]['ch'])

            for _to_, index_list in dic_trigger_index.items():
                for index in index_list:

                    cd1 = merged_dict.get(_to_, {}).get(apa1, {}).get("channel_dic")
                    cd2 = merged_dict.get(_to_, {}).get(apa2, {}).get("channel_dic")

                    ch_data1 = None
                    ch_data2 = None

                    if isinstance(cd1, list) and index < len(cd1):
                        ch_data1 = cd1[index].get(end1, {}).get(ch1)

                    if isinstance(cd2, list) and index < len(cd2):
                        ch_data2 = cd2[index].get(end2, {}).get(ch2)

                    if ch_data1 and ch_data2:
                        
                        n_pe1 = ch_data1['n_pe']
                        e_n_pe1 = ch_data1['e_n_pe']
                        n_pe2 = ch_data2['n_pe']
                        e_n_pe2 = ch_data2['e_n_pe']

                        ch_pair_info[0]['NPE'].append(n_pe1)
                        ch_pair_info[0]['NPE_err'].append(e_n_pe1)
                        ch_pair_info[1]['NPE'].append(n_pe2)
                        ch_pair_info[1]['NPE_err'].append(e_n_pe2)
            
            
            for i in range(2):
                ch_pair_info[i]['NPE_mean'] = np.mean(ch_pair_info[i]['NPE']) if ch_pair_info[i]['NPE'] else 0
                ch_pair_info[i]['NPE_std'] = np.std(ch_pair_info[i]['NPE']) if ch_pair_info[i]['NPE'] else 0
                ch_pair_info[i]['NPE_err_mean'] = ch_pair_info[i]['NPE_mean'] / np.sqrt(len(ch_pair_info[i]['NPE'])) if ch_pair_info[i]['NPE'] else 0
            

            adiacent_channels_apa_list.append(ch_pair_info)
        
        adiacent_channels_dict[apa] = adiacent_channels_apa_list
        
    all_data[energy] = adiacent_channels_dict
    


In [ ]:
colors = {1: "red", 2: "green", 3: "blue", 5: "orange", 7: "purple"}

energies = np.array(energies)

strategies = [
    'N1-N2',
    '(N1-N2)/((N1+N2)/2)'
]


In [ ]:
adiacent_channels_map = {'1': [[{'apa' : 1, 'end': 105, 'ch': 15},{'apa' : 1, 'end': 105, 'ch': 17}]]}

In [ ]:
APA_pdf_file = PdfPages(f"/afs/cern.ch/work/a/anbalbon/private/waffles/src/waffles/np04_analysis/lightyield_vs_energy/output/adjacent_chanels_study/multi_pair_analysis.pdf")
all_results = []
for apa, adiacent_channels_apa in adiacent_channels_map.items():
    for ch_pairs in adiacent_channels_apa:
        fig, axes = plt.subplots(len(strategies)+1, len(energies)+1, figsize=(6*len(energies), 20))
        
        
        all_mu       = {s: [] for s in strategies}
        all_mu_err   = {s: [] for s in strategies}
        all_sigma        = {s: [] for s in strategies}
        all_sigma_err    = {s: [] for s in strategies}
        all_mean = {s: [] for s in strategies}
        all_std = {s: [] for s in strategies}
        all_mean_pe = []
        all_mean_pe_err = []

        scatter_plot_results = {}
        valid_energies = []
        for j, energy in enumerate(energies):
            valid_energy = False

            ax = axes[0, j]

            for data_ch_pair in all_data[energy][apa]:
                if (data_ch_pair[0]['end'] == ch_pairs[0]['end'] and data_ch_pair[0]['ch'] == ch_pairs[0]['ch'])  and (data_ch_pair[1]['end'] == ch_pairs[1]['end'] and data_ch_pair[1]['ch'] == ch_pairs[1]['ch']):
                    x = np.array(data_ch_pair[0]['NPE'])
                    ex = np.array(data_ch_pair[0]['NPE_err'])
                    y = np.array(data_ch_pair[1]['NPE'])
                    ey = np.array(data_ch_pair[1]['NPE_err'])

                    if len(x) < 500:
                        print(f"Warning: Only {len(x)} events for energy {energy} GeV, APA {apa}, channels {ch_pairs[0]['end']}ch{ch_pairs[0]['ch']} and {ch_pairs[1]['end']}ch{ch_pairs[1]['ch']}. Results might not be reliable.")
                        continue
                    else:
                        valid_energies.append(energy)
                        valid_energy = True

                    all_mean_pe.append(data_ch_pair[0]['NPE_mean'])
                    all_mean_pe_err.append(data_ch_pair[0]['NPE_err_mean'])

                    
                    ax.errorbar(
                        x, y,
                        xerr=ex, yerr=ey,
                        fmt='o',
                        markersize=5,
                        capsize=3,
                        elinewidth=1,
                        markeredgecolor='black',
                        markerfacecolor=colors[energy],
                        label="Data"
                    )

                    xmin = np.minimum(np.min(x), np.min(y)) - 50
                    xmax = np.maximum(np.max(x), np.max(y)) + 50

                    # Reference line y = x
                    ax.plot([xmin, xmax], [xmin, xmax], 'k--', label='Expected y = x')

                    # ----- ODR linear fit -----
                    data = RealData(x, y, sx=ex, sy=ey)
                    model = Model(linear_array)
                    odr = ODR(data, model, beta0=curve_fit(linear, x, y, sigma=ey, absolute_sigma=True)[0])
                    out = odr.run()

                    A, B = out.beta
                    eA, eB = out.sd_beta

                    y_fit = linear(x, A, B)

                    r_squared = r2_score(y, y_fit)
                    chi2rid = out.sum_square / (len(x) - len(out.beta))

                    label_fit = (
                        f'y = A + Bx\n'
                        f'A = ({fmt(A, eA)})\n'
                        f'B = ({fmt(B, eB)})\n'
                        f'$R^2$ = {r_squared:.3f}\n'
                        f'$\chi^2_{{rid}}$ = {chi2rid:.2f}'
                    )

                    ax.plot([xmin, xmax],
                            linear_array([A,B], np.array([xmin, xmax])),
                            'r-', label=label_fit)
                    
                    scatter_plot_results[energy] = {'A': (A, eA), 'B': (B, eB), 'r_squared': r_squared, 'chi2rid': chi2rid}

            if valid_energy:
                ax.set_xlim(xmin, xmax)
                ax.set_ylim(xmin, xmax)
                ax.legend(loc='upper left', fontsize=8)
            else:
                ax.text(0.5, 0.5,
                        "No valid data",
                        transform=ax.transAxes,
                        ha='center', va='center')
                ax.set_axis_off()

            ax.set_xlabel(
                f"$N_{{PE}}$ - end{ch_pairs[0]['end']}ch{ch_pairs[0]['ch']}",
                fontsize=11
            )
            ax.set_ylabel(
                f"$N_{{PE}}$ - end{ch_pairs[1]['end']}ch{ch_pairs[1]['ch']}",
                fontsize=11
            )
            ax.set_title(f"{energy} GeV", fontsize=13)
            ax.grid(True)
            
            
            if not valid_energy:
                continue

            for i, residual_strategy in enumerate(strategies):

                # HISTOGRAM OF RESIDUALS
                ax = axes[i+1, j]
                if residual_strategy =='N1-N2':
                    residuals =  x-y 
                    label_x = r"$N_1 - N_2$"
                
                elif residual_strategy == '(N1-N2)/((N1+N2)/2)':
                    residuals = (x-y)/((x+y)/2)
                    label_x = r"$\frac{N_1 - N_2}{(N_1 + N_2) /2}$"

                else:
                    continue

                N = len(residuals)
                mean_0 = np.mean(residuals)
                std_0  = np.std(residuals, ddof=1)

                Range = [np.percentile(residuals, 0.3), np.percentile(residuals, 99.8)]
                bins = 50

                ax.hist(
                    residuals,
                    bins=bins,
                    range=Range,
                    color=colors[energy],
                    alpha=0.8,
                    label=f"N = {N}\n$\mu$ = {mean_0:.2}\n$\sigma$ = {std_0:.2f}"
                )

                # ===== Gaussian fit of histogram =====
                counts, bin_edges = np.histogram(residuals, bins=bins, range=Range)
                bin_centers = 0.5 * (bin_edges[:-1] + bin_edges[1:])
                bin_width = bin_edges[1] - bin_edges[0]

                mask = counts > 0
                x_fit = bin_centers[mask]
                y_fit = counts[mask]

                sx = np.full_like(x_fit, bin_width / 2)
                sy = np.sqrt(y_fit)

                data_g = RealData(x_fit, y_fit, sx=sx, sy=sy)
                model_g = Model(gaussian_array)

                odr_g = ODR(data_g, model_g, beta0=[mean_0, std_0, np.max(y_fit)])
                out_g = odr_g.run()

                mu, sigma, A_g = out_g.beta
                emu, esigma, eA_g = out_g.sd_beta

                y_model_g = gaussian(x_fit, mu, sigma, A_g)
                r_squared = r2_score(y_fit, y_model_g)
                chi2rid = out_g.sum_square / (len(x_fit)-len(out_g.beta))

                label_fit = (
                    f"$\mu$ = {fmt(mu, emu)}\n"
                    f"$\sigma$ = {fmt(sigma, esigma)}\n"
                    f"A = {fmt(A_g,eA_g)}\n"
                    f"R$^2$ = {r_squared:.3f}\n"
                    f"$\chi^2_{{rid}}$ = {chi2rid:.3f}"
                )

                x_plot = np.linspace(Range[0], Range[1], 500)
                y_plot = gaussian(x_plot, mu, sigma, A_g)

                ax.plot(x_plot, y_plot, 'k-', linewidth=2, label=label_fit)
                ax.axvline(mu, linestyle='--')

                ax.set_title(f"{energy} GeV — {residual_strategy}", fontsize=13)
                ax.set_ylabel("Counts", fontsize=11)
                ax.set_xlabel(label_x, fontsize=11)

                ax.legend(loc="upper left", fontsize=8)
                ax.grid(True)

                all_mu[residual_strategy].append(mu)
                all_mu_err[residual_strategy].append(emu)
                all_sigma[residual_strategy].append(sigma)
                all_sigma_err[residual_strategy].append(esigma)
                all_mean[residual_strategy].append(mean_0)
                all_std[residual_strategy].append(std_0)
            

        # # ENERGY RESOLUTION
        energy_resolution_result = {}

        

        
        n_valid = len(all_mean_pe)
        if n_valid < 3:
            print(f"Skipping energy resolution: only {n_valid} valid energies")

            for i in range(len(strategies)):
                ax = axes[i+1, -1]
                ax.text(0.5, 0.5,
                        f"Not enough energies\n({n_valid} < 3)",
                        transform=ax.transAxes,
                        ha='center', va='center')
                ax.set_axis_off()

        else:
            for i, residual_strategy in enumerate(strategies):
                

                ax = axes[i+1, -1]

                x = np.array(valid_energies)
                ex = 0.05*x

                all_sigma[residual_strategy] = np.array(all_sigma[residual_strategy])
                all_sigma_err[residual_strategy] = np.array(all_sigma_err[residual_strategy])
                all_mean_pe = np.array(all_mean_pe)
                all_mean_pe_err = np.array(all_mean_pe_err)

                if residual_strategy == 'N1-N2':
                    y = (1/np.sqrt(2))* (all_sigma[residual_strategy]/all_mean_pe)
                    ey = y * np.sqrt((all_sigma_err[residual_strategy]/all_sigma[residual_strategy])**2+(all_mean_pe_err/all_mean_pe)**2)
                    y_label = r'$\frac{1}{\sqrt{2}} \cdot \sigma(N_1 - N_2) \cdot \frac{1}{\langle N_1 \rangle}$'
                elif residual_strategy == '(N1-N2)/((N1+N2)/2)':
                    y = (1/np.sqrt(2))* all_sigma[residual_strategy]
                    ey = (1/np.sqrt(2))* all_sigma_err[residual_strategy]
                    y_label = r'$\frac{1}{\sqrt{2}} \cdot \sigma\!\left(\frac{N_1 - N_2}{(N_1 + N_2) /2}\right)$'
                else:
                    continue
                    

                ax.errorbar(x,
                    y,
                    xerr = ex,
                    yerr = ey,
                    marker='o',
                    linestyle='',
                    markersize=4,
                    linewidth=1,
                    capsize=3,
                    label=f"Data"
                    )
                        
                    
                p0_init = np.min(y)                  
                p1_init = np.sqrt(np.maximum(y[0]**2 - p0_init**2, 1e-6)) * np.sqrt(x[0])            
                p2_init = 0.1                       
                
                data = RealData(x, y, sx=ex, sy=ey)
                model = Model(energy_resolution_fit_array)

                odr = ODR(data, model, beta0=np.array([p0_init, p1_init, p2_init]))
                odr.set_job(fit_type=0)
                out = odr.run()

                p0, p1, p2 = out.beta
                ep0, ep1, ep2 = out.sd_beta

                y_fit = energy_resolution_fit(x,p0,p1,p2)
                chi2_red = out.sum_square / (len(x) - len(out.beta)) 
                r_squared = r2_score(y,y_fit)
        
                y_fit_label = (f"Fit "
                r'$y=\sqrt{p_0^2 + \left(\frac{p_1}{\sqrt{x}}\right)^2 + \left(\frac{p_2}{x}\right)^2}$'
                f'\n'
                f"p0 = {fmt(p0,ep0)}\n"
                f"p1 = {fmt(p1,ep1)}\n"
                f"p2 = {fmt(p2,ep2)}\n"
                f"$R^2$ = {r_squared:.3f}\n"
                r"$\chi^2_{rid}$" +f" = {chi2_red:.3f}")

                x_plot = np.linspace(np.min(x), np.max(x), 500)
                y_plot = energy_resolution_fit(x_plot, p0, p1, p2)
                ax.plot(x_plot, y_plot, label=y_fit_label)

                ax.set_xlabel(r"$\langle E_{beam} \rangle$ [GeV]")
                ax.set_ylabel(y_label)
                ax.set_title(f"Energy Resolution", fontsize=13)
                ax.grid(True)
                ax.set_xlim(0, 8)
                ax.legend(fontsize=9)

                energy_resolution_result[residual_strategy] = {'x': x, 'ex': ex,
                    'y': y,
                    'ey': ey,
                    'fit_params': {
                        'p0': (p0, ep0),
                        'p1': (p1, ep1),
                        'p2': (p2, ep2)},
                    'chi2_red': chi2_red,
                    'r_squared': r_squared
                    }


    
            APA_pdf_file.savefig(fig)
        plt.show()
        plt.close(fig)


        ch_pair_result = {
            'apa': apa,
            'channel_pair': ch_pairs,
            'energies': energies,
            'strategies': strategies,
            'all_mu': all_mu,
            'all_mu_err': all_mu_err,
            'all_sigma': all_sigma,
            'all_sigma_err': all_sigma_err,
            'all_mean': all_mean,
            'all_std': all_std,
            'all_mean_pe': all_mean_pe,
            'all_mean_pe_err': all_mean_pe_err,
            'scatter_plot' : scatter_plot_results,
            'energy_resolution': energy_resolution_result
        }

        all_results.append(ch_pair_result)

APA_pdf_file.close()


In [ ]:
# adiacent_channels_map = {'1': [[{'apa' : 1, 'end': 104, 'ch': 15},{'apa' : 1, 'end': 104, 'ch': 12}], [{'apa' : 1, 'end': 104, 'ch': 2},{'apa' : 1, 'end': 104, 'ch': 0}],[{'apa' : 1, 'end': 104, 'ch': 3},{'apa' : 1, 'end': 104, 'ch': 4}]],
# '2' : [[{'apa' : 2, 'end': 109, 'ch': 37},{'apa' : 2, 'end': 109, 'ch': 35}], [{'apa' : 2, 'end': 109, 'ch': 31},{'apa' : 2, 'end': 109, 'ch': 33}], [{'apa' : 2, 'end': 109, 'ch': 34},{'apa' : 2, 'end': 109, 'ch': 36}], [{'apa' : 2, 'end': 109, 'ch': 2},{'apa' : 2, 'end': 109, 'ch': 0}] ]}    

# adiacent_channels_1 = [[{'apa' : 1, 'end': 104, 'ch': 7},{'apa' : 1, 'end': 104, 'ch': 5}], # 1
#                         [{'apa' : 1, 'end': 104, 'ch': 5},{'apa' : 1, 'end': 104, 'ch': 2}], # 1
#                         [{'apa' : 1, 'end': 104, 'ch': 2},{'apa' : 1, 'end': 104, 'ch': 0}], # 1
#                         [{'apa' : 1, 'end': 104, 'ch': 1},{'apa' : 1, 'end': 104, 'ch': 3}], # 2
#                         [{'apa' : 1, 'end': 104, 'ch': 3},{'apa' : 1, 'end': 104, 'ch': 4}], # 2
#                         [{'apa' : 1, 'end': 104, 'ch': 4},{'apa' : 1, 'end': 104, 'ch': 6}], # 2
#                         [{'apa' : 1, 'end': 104, 'ch': 17},{'apa' : 1, 'end': 104, 'ch': 15}], # 3
#                         [{'apa' : 1, 'end': 104, 'ch': 15},{'apa' : 1, 'end': 104, 'ch': 12}], # 3
#                         [{'apa' : 1, 'end': 104, 'ch': 12},{'apa' : 1, 'end': 104, 'ch': 10}], # 3
#                         [{'apa' : 1, 'end': 104, 'ch': 11},{'apa' : 1, 'end': 104, 'ch': 13}], # 4
#                         [{'apa' : 1, 'end': 104, 'ch': 13},{'apa' : 1, 'end': 104, 'ch': 14}], # 4
#                         [{'apa' : 1, 'end': 104, 'ch': 14},{'apa' : 1, 'end': 104, 'ch': 16}], # 4
#                         [{'apa' : 1, 'end': 105, 'ch': 7},{'apa' : 1, 'end': 105, 'ch': 5}], # 5 
#                         [{'apa' : 1, 'end': 105, 'ch': 5},{'apa' : 1, 'end': 105, 'ch': 2}], # 5 
#                         [{'apa' : 1, 'end': 105, 'ch': 2},{'apa' : 1, 'end': 105, 'ch': 0}], # 5 
#                         [{'apa' : 1, 'end': 105, 'ch': 1},{'apa' : 1, 'end': 105, 'ch': 3}], # 6
#                         [{'apa' : 1, 'end': 105, 'ch': 3},{'apa' : 1, 'end': 105, 'ch': 4}], # 6 
#                         [{'apa' : 1, 'end': 105, 'ch': 4},{'apa' : 1, 'end': 105, 'ch': 6}], # 6 
#                         [{'apa' : 1, 'end': 105, 'ch': 26},{'apa' : 1, 'end': 105, 'ch': 24}], # 7
#                         [{'apa' : 1, 'end': 105, 'ch': 24},{'apa' : 1, 'end': 105, 'ch': 23}], # 7 
#                         [{'apa' : 1, 'end': 105, 'ch': 23},{'apa' : 1, 'end': 105, 'ch': 21}], # 7 
#                         [{'apa' : 1, 'end': 105, 'ch': 10},{'apa' : 1, 'end': 105, 'ch': 12}], # 8
#                         [{'apa' : 1, 'end': 105, 'ch': 12},{'apa' : 1, 'end': 105, 'ch': 15}], # 8 
#                         [{'apa' : 1, 'end': 105, 'ch': 15},{'apa' : 1, 'end': 105, 'ch': 17}], # 8 
#                         ]

#     adiacent_channels_2 = [[{'apa' : 2, 'end': 109, 'ch': 27},{'apa' : 2, 'end': 109, 'ch': 25}], # 1 
#                         [{'apa' : 2, 'end': 109, 'ch': 25},{'apa' : 2, 'end': 109, 'ch': 22}], # 1 
#                         [{'apa' : 2, 'end': 109, 'ch': 22},{'apa' : 2, 'end': 109, 'ch': 20}], # 1 
#                         [{'apa' : 2, 'end': 109, 'ch': 21},{'apa' : 2, 'end': 109, 'ch': 23}], # 2
#                         [{'apa' : 2, 'end': 109, 'ch': 23},{'apa' : 2, 'end': 109, 'ch': 24}], # 2 
#                         [{'apa' : 2, 'end': 109, 'ch': 24},{'apa' : 2, 'end': 109, 'ch': 26}], # 2
#                         [{'apa' : 2, 'end': 109, 'ch': 37},{'apa' : 2, 'end': 109, 'ch': 35}], # 3 
#                         [{'apa' : 2, 'end': 109, 'ch': 35},{'apa' : 2, 'end': 109, 'ch': 32}], # 3 
#                         [{'apa' : 2, 'end': 109, 'ch': 32},{'apa' : 2, 'end': 109, 'ch': 30}], # 3
#                         [{'apa' : 2, 'end': 109, 'ch': 31},{'apa' : 2, 'end': 109, 'ch': 33}], # 4 
#                         [{'apa' : 2, 'end': 109, 'ch': 33},{'apa' : 2, 'end': 109, 'ch': 34}], # 4 
#                         [{'apa' : 2, 'end': 109, 'ch': 34},{'apa' : 2, 'end': 109, 'ch': 36}], # 4   
#                         [{'apa' : 2, 'end': 109, 'ch': 7},{'apa' : 2, 'end': 109, 'ch': 5}], # 5 
#                         [{'apa' : 2, 'end': 109, 'ch': 5},{'apa' : 2, 'end': 109, 'ch': 2}], # 5 
#                         [{'apa' : 2, 'end': 109, 'ch': 2},{'apa' : 2, 'end': 109, 'ch': 0}], # 5 
#                         [{'apa' : 2, 'end': 109, 'ch': 1},{'apa' : 2, 'end': 109, 'ch': 3}], # 6
#                         [{'apa' : 2, 'end': 109, 'ch': 3},{'apa' : 2, 'end': 109, 'ch': 4}], # 6 
#                         [{'apa' : 2, 'end': 109, 'ch': 4},{'apa' : 2, 'end': 109, 'ch': 6}], # 6
#                         [{'apa' : 2, 'end': 109, 'ch': 17},{'apa' : 2, 'end': 109, 'ch': 15}], # 7 
#                         [{'apa' : 2, 'end': 109, 'ch': 15},{'apa' : 2, 'end': 109, 'ch': 12}], # 7 
#                         [{'apa' : 2, 'end': 109, 'ch': 12},{'apa' : 2, 'end': 109, 'ch': 10}], # 7
#                         [{'apa' : 2, 'end': 109, 'ch': 11},{'apa' : 2, 'end': 109, 'ch': 13}], # 8 
#                         [{'apa' : 2, 'end': 109, 'ch': 13},{'apa' : 2, 'end': 109, 'ch': 14}], # 8 
#                         [{'apa' : 2, 'end': 109, 'ch': 14},{'apa' : 2, 'end': 109, 'ch': 16}], # 8
#                         [{'apa' : 2, 'end': 109, 'ch': 47},{'apa' : 2, 'end': 109, 'ch': 45}], # 9 
#                         [{'apa' : 2, 'end': 109, 'ch': 45},{'apa' : 2, 'end': 109, 'ch': 42}], # 9 
#                         [{'apa' : 2, 'end': 109, 'ch': 42},{'apa' : 2, 'end': 109, 'ch': 40}], # 9 
#                         [{'apa' : 2, 'end': 109, 'ch': 41},{'apa' : 2, 'end': 109, 'ch': 43}], # 10
#                         [{'apa' : 2, 'end': 109, 'ch': 43},{'apa' : 2, 'end': 109, 'ch': 44}], # 10
#                         [{'apa' : 2, 'end': 109, 'ch': 44},{'apa' : 2, 'end': 109, 'ch': 46}], # 10  
#                         ]

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

pair_labels = []
A_mean = []
A_err = []
B_mean = []
B_err = []

for result in all_results:

    ch1 = result['channel_pair'][0]
    ch2 = result['channel_pair'][1]

    label = f"end{ch1['end']}ch{ch1['ch']} — end{ch2['end']}ch{ch2['ch']}"
    
    A_vals = []
    A_errs = []
    B_vals = []
    B_errs = []

    for energy, fit in result['scatter_plot'].items():
        A_vals.append(fit['A'][0])
        A_errs.append(fit['A'][1])
        B_vals.append(fit['B'][0])
        B_errs.append(fit['B'][1])

    if len(A_vals) == 0:
        continue

    # media pesata
    weights_A = 1 / np.array(A_errs)**2
    weights_B = 1 / np.array(B_errs)**2

    A_wmean = np.sum(weights_A * A_vals) / np.sum(weights_A)
    B_wmean = np.sum(weights_B * B_vals) / np.sum(weights_B)

    A_werr = np.sqrt(1 / np.sum(weights_A))
    B_werr = np.sqrt(1 / np.sum(weights_B))

    pair_labels.append(label)
    A_mean.append(A_wmean)
    A_err.append(A_werr)
    B_mean.append(B_wmean)
    B_err.append(B_werr)


plt.figure(figsize=(12,6))
plt.errorbar(pair_labels, A_mean, yerr=A_err, fmt='o')
plt.xticks(rotation=90)
plt.ylabel("A (weighted mean)")
plt.title("Linear fit parameter A per channel pair")
plt.grid(True)
plt.tight_layout()
plt.show()

plt.figure(figsize=(12,6))
plt.errorbar(pair_labels, B_mean, yerr=B_err, fmt='o')
plt.xticks(rotation=90)
plt.ylabel("B (weighted mean)")
plt.title("Linear fit parameter B per channel pair")
plt.grid(True)
plt.tight_layout()
plt.show()


p0_vals = []
p0_errs = []
p1_vals = []
p1_errs = []
p2_vals = []
p2_errs = []
res_labels = []

for result in all_results:

    if len(result['energy_resolution']) == 0:
        continue

    ch1 = result['channel_pair'][0]
    ch2 = result['channel_pair'][1]

    label = f"end{ch1['end']}ch{ch1['ch']} — end{ch2['end']}ch{ch2['ch']}"

    # prendo la prima strategy (oppure scegli quella che ti interessa)
    fit = list(result['energy_resolution'].values())[0]

    p0, ep0 = fit['fit_params']['p0']
    p1, ep1 = fit['fit_params']['p1']
    p2, ep2 = fit['fit_params']['p2']

    res_labels.append(label)

    p0_vals.append(p0)
    p0_errs.append(ep0)

    p1_vals.append(p1)
    p1_errs.append(ep1)

    p2_vals.append(p2)
    p2_errs.append(ep2)

plt.figure(figsize=(12,6))
plt.errorbar(res_labels, p0_vals, yerr=p0_errs, fmt='o')
plt.xticks(rotation=90)
plt.ylabel("p0")
plt.title("Energy resolution parameter p0")
plt.grid(True)
plt.tight_layout()
plt.show()